## Fact Sales 

Fato de vendas na camada Silver, integrada às dimensões analíticas.

In [0]:
# Imports
from pyspark.sql import functions as F

In [0]:
# Leitura das tabelas Silver 

df_sales_header = spark.table("adventure_works_catalog.silver.clean_sales_order_header")
df_sales_detail = spark.table("adventure_works_catalog.silver.clean_sales_order_detail")

# Leitura das Dimensões
df_dim_product = spark.table("adventure_works_catalog.silver.dim_product")
df_dim_customer = spark.table("adventure_works_catalog.silver.dim_customer")
df_dim_date = spark.table("adventure_works_catalog.silver.dim_date")
df_dim_currency = spark.table("adventure_works_catalog.silver.dim_currency")


# Criação de coluna para comentários e documentação
def add_column_comments(catalog, schema, table, columns_dict):
    for column, comment in columns_dict.items():
        spark.sql(f"""
            ALTER TABLE `{catalog}`.`{schema}`.`{table}`
            ALTER COLUMN `{column}`
            COMMENT '{comment}'
        """)


In [0]:
# ============================================================
# FACT_SALES — SILVER (DIMENSIONAL)
# ============================================================

from pyspark.sql import functions as F




# Join Header + Detail
df_sales_base = (
    df_sales_detail.alias("sd")
    .join(
        df_sales_header.alias("sh"),
        on="SalesOrderID",
        how="inner"
    )
)


# Join com Dimensões 
df_sales_enriched = (
    df_sales_base

    # Product
    .join(
        df_dim_product.alias("p"),
        F.col("sd.ProductID") == F.col("p.ProductID"),
        "left"
    )

    # Customer
    .join(
        df_dim_customer.alias("c"),
        F.col("sh.CustomerID") == F.col("c.CustomerID"),
        "left"
    )

    # Order Date
    .join(
        df_dim_date.alias("od"),
        F.to_date(F.col("sh.OrderDate")) == F.col("od.FullDate"),
        "left"
    )

    # Ship Date
    .join(
        df_dim_date.alias("sdte"),
        F.to_date(F.col("sh.ShipDate")) == F.col("sdte.FullDate"),
        "left"
    )

    # Currency
    .join(
        df_dim_currency.alias("cur"),
        F.col("sh.CurrencyRateID") == F.col("cur.CurrencyRateID"),
        "left"
    )
)


# Seleção final 
df_fact_sales = (
    df_sales_enriched
    .select(
        # Identificadores
        F.col("sd.SalesOrderDetailID").alias("SalesID"),
        F.col("sd.SalesOrderID").alias("OrderID"),

        # Chaves substitutas
        F.col("p.Product_SK").alias("Product_SK"),
        F.col("c.Customer_SK").alias("Customer_SK"),
        F.col("od.DateKey").alias("OrderDate_SK"),
        F.col("sdte.DateKey").alias("ShipDate_SK"),
        F.col("cur.Currency_SK").alias("Currency_SK"),

        # Métricas
        F.col("sd.OrderQty").alias("OrderQty"),
        F.col("sd.UnitPrice"),
        F.col("sd.UnitPriceDiscount"),
        F.col("sd.LineTotal"),
        F.col("sh.Freight").alias("ShippingCost"),
        F.col("sh.TaxAmt").alias("TaxAmt")
    )
)


# Garantia de grão
df_fact_sales = df_fact_sales.dropDuplicates(
    ["SalesID", "OrderID"]
)

# Conferência
df_fact_sales.printSchema()
df_fact_sales.display()


# Escrita na Silver
(
    df_fact_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("adventure_works_catalog.silver.fact_sales")
)

# Descrição das colunas
fact_sales_columns = {
    "SalesID": "Identificador único do item da venda",
    "OrderID": "Identificador do pedido de venda",
    "Product_SK": "Chave substituta do produto vendido",
    "Customer_SK": "Chave substituta do cliente",
    "OrderDate_SK": "Chave da data do pedido de venda",
    "ShipDate_SK": "Chave da data de envio do pedido",
    "Currency_SK": "Chave substituta da moeda utilizada na venda",
    "OrderQty": "Quantidade vendida do produto",
    "UnitPrice": "Preço unitário do produto na venda",
    "UnitPriceDiscount": "Desconto aplicado sobre o preço unitário",
    "LineTotal": "Valor total do item vendido",
    "ShippingCost": "Custo de frete do pedido de venda",
    "TaxAmt": "Valor de imposto aplicado ao pedido de venda"
}

# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="fact_sales",
    columns_dict=fact_sales_columns
)
